
   # Testing API calls in Python
## Duncan Leggat     

 ### 2026-04-21

# Duncan Leggat - A brief history


- Studied physics (MPhys)

  <img
    data-fragment-index="0"
    src="./images/soton_logo.png"
    width="200"
  />

- Worked in particle physics!

  <img
    src="./images/hep_logos.png"
  />

 - Here are some HEP plots!
 
  <img
      class="fragment"
      src="./images/tw_obs.png"
      />

- Postdoc at University of Leeds - in Performance and Creative Industries!


  <img
    src="./images/leeds_logo.png"
       width="200"
  />
  <img
       class="fragment"
       src="./images/song_future.png"
       />

## Outside of work

- Write prose! [Check out my SubStack](https://substack.com/@duncwritesthings)

- Write music! [Check out my SoundCloud](https://soundcloud.com/dacacia)

- Playing games! Challenge me at Smash Bros!

![](./images/animals_new_1.png)

# Motivation



- Chatting to David about the portfolio management tools and how we test the API calls there offline.
- He hadn't heard of the `VCR` package for pytest, so he suggested giving a talk to it.
- Then I decided it would be interesting to give a talk about the various testing methods we use in the portfolio to handle API calls, and changing the states of the websites we poll.
- Then I thought it would be cool to have live-coding/demonstrations built into the talk. 

- So anyway, now it's basically a tutorial.

### Enjoy?

# API calls in Python

## Some definitions (or - what do you mean by API calls?)

**API** - Application Programming Interface

Defines how to interface with a piece of software.

Common example: An ATM is an API between the user and the bank's systems.

**API calls**

In this talk, 'API calls' will refer to specifically Hypertext Transfer Protocol (HTTP) requests, the standard method for interacting over networks.

## What is an HTTP request?

A standard HTTP request consists of 5 parts:
- **Request type**: What type of request is being made. Examples include: `GET`, `PUT`, `PATCH`, or `DELETE`.
- **Endpoint URL**: The address of the resource being requested.
- **Headers**: Metadata of the request, which may include authentication if required.
- **Request body**: The data required for requests such as `PUT` and `PATCH`.
- **Query parameters**: Filters on the request (the values after a `?` in a URL)

## Let's make some API calls in Python

The [`requests`](https://pypi.org/project/requests/) module handles HTTP requests in Python.

In [ ]:
import requests

Let's make some API calls:

In [ ]:
# Target URL
url = "https://httpbin.org/" # Website that allows test API calls

# Make a request
response = requests.get(url)

# Print the response code
print(response)

## Making a `POST` request

In [ ]:
# A post request requires a payload
payload = {"test_post":"test!"}

url_post = "https://httpbin.org/anything" # Bounces back the data sent

response = requests.post(url_post,data=payload)
print(response)
print(response.text)
#print(response.json()["form"])

## Making API calls to GitHub

Demo project board for this presentation:
![image](./images/project_board.png)

To get the issues in this repository:
1) To access GitHub, we need to get authorisation using a GitHub token in the request headers

In [ ]:
def get_github_token(secrets_file):
    ### Get github token from secrets file
    github_token = ""
    import yaml
    from pathlib import Path
    with Path(secrets_file).open() as f:
        file_contents = yaml.safe_load(f)
        if "github_token" in file_contents.keys():
            return file_contents["github_token"]

github_token = get_github_token("secrets.yml")

def get_auth_headers():
    ### Get the authorisation headers required for GitHub access

    headers = {"Authorization": f"Bearer {github_token}"}
    return {"headers": headers}

2) Use [REST API](https://docs.github.com/en/rest) end points for GitHub to retrieve repository issues

In [ ]:
def get_repo_issues(repo: str, state="all"):
    ### Get all issues from repo in given state
    
    # REST API endpoint for issues in a repo.
    # repo has form owner/repo
    url = f"https://api.github.com/repos/{repo}/issues"
    
    # Accessing GitHub requires authorisation
    kwargs = get_auth_headers()

    # Filter based on chosen state
    kwargs["params"] = {"state": state}

    return requests.get(url, **kwargs)
    
# My testing repo
repo = "dleggat/api_testing"
response = get_repo_issues(repo)

print(response)
data = response.json()

3) Print the issues

In [ ]:
print(data)

In [ ]:
def print_issues(data):
    ### Print issues in a human readable way
    print("# | ? | Name")
    print("------------")
    for datum in data:
        print(f"{datum['number']} | {'✓' if datum['state'] == 'open' else 'x'} | {datum['title']}")

print_issues(data)

But we've opened multiple issues, so that's complete! 

Let's close the issue from another API call:

In [ ]:
import json

def patch_issue(repo, issue_num, payload):
    ### Patch issue issue_num in repo with payload

    # REST API endpoint for a single issue
    url_patch = f"https://api.github.com/repos/{repo}/issues/{issue_num}"
    
    # Move our changes into a JSON format
    data = json.dumps(payload)
    
    # Get authorisation for GitHub
    kwargs = get_auth_headers()
    
    return requests.patch(url_patch, data=data, **kwargs)

repo = "dleggat/api_testing"
payload = {"state": "open"}
issue_num = 2
response_patch = patch_issue(repo, issue_num, payload)

print(response_patch)

In [ ]:
def print_repo_issues(repo):
    ### Combine the get_repo_issues and print_issues methods
    
    print_issues(get_repo_issues(repo).json())

print_repo_issues(repo)

## **A short aside** - running `pytest` in Jupyter using `ipytest`

In [ ]:
import ipytest
import pytest

ipytest.autoconfig()

def double_number(number):
    ### Double the number
    return 2*number

class TestCase:
    @pytest.mark.parametrize(
        ["in_num","output"],
        [
            [2,4],
            [3,6],
            [4,8],
            [5,10],
        ]
    )
    def test_double_number(self,in_num,output):
        assert output == double_number(in_num)
    
    def test_issues(self):
        response = get_repo_issues("dleggat/api_testing")
        assert len(response.json()) == 4

ipytest.run()

But... when testing, we don't actually want to be making real API calls! 
We want our tests to be repeatable, and not beholden to the state of the service we're calling - we want to be offline!

# Testing API calls in pytest

## Approach 1 - Using the `vcr` package

One way to handle API calls in testing is the [`vcr`] (https://pypi.org/project/pytest-recording/)(now in `pytest-recording`) package.

In [ ]:
%%file vcr_test.py

import pytest
import api_calls

class TestCase:
    @pytest.mark.vcr
    def test_use_vcr(self):
        test_repo = "dleggat/api_testing"
        response = api_calls.get_repo_issues(test_repo)
        assert len(response.json()) == 4
        assert response.status_code == 200
        
        response = api_calls.get_repo_issues("not_real/fake")
        
        assert response.status_code == 404

In [ ]:
!pytest --record-mode=once vcr_test.py

In [ ]:
# Here in case I mess things up and need to reset the VCR - hopefully I won't run this cell!
# !rm cassettes/vcr_test/TestCase.test_use_vcr.yaml

The first time this runs, it produces a 'cassette' file: `cassettes/{test_file_name}/{text_class_name}.{test_name}.yaml`

For this test, it is therefore called [`cassettes/vcr_test/TestCase.test_use_vcr.yaml`](./cassettes/vcr_test/TestCase.test_use_vcr.yaml)

In [ ]:
!ls -la cassettes/vcr_test/TestCase.test_use_vcr.yaml

In [ ]:
!cat cassettes/vcr_test/TestCase.test_use_vcr.yaml | grep -v "Bear"

Now, if we run the test again, no actual API call will be made - we will instead refer to this saved output.

Let's see what happens if we play around with the cassette data:

In [ ]:
!sed 's/code: 200/code: 202/' -i cassettes/vcr_test/TestCase.test_use_vcr.yaml

## Cassettes - pros and cons

Why **should** you use `vcr` cassettes?
- Let's you use the *real* data!
- Response is fixed in time, so will always be consistent and repeatable.

Why **shouldn't** you use them?
 - They can be cumbersome.
 - Not the most transparent objects.
 - If you ever need more information from the resource, the cassettes need to be re-recorded, and this might break older tests.

## Approach 2 - Mock the calls

- Use `patch` fixture to replace the function that carries out the API call

In [ ]:
from unittest.mock import patch

class TestCase:
    # Makes any call to requests.get a blank call instead
    @patch("requests.get")
    def test_patch_implementation(self, mock_get):
        test_repo = "leggat/d"
        get_repo_issues(test_repo)
        
        # Some tests we can call for the mocked function
        mock_get.assert_called()
        mock_get.assert_called_once()
        
ipytest.run()

In addition to testing that the mock function is called, we can check the arguments used:

```py
expected_url = f"https://api.github.com/repos/{test_repo}/issues"
kwargs = get_auth_headers()
kwargs["params"] = {"state": "all"}
mock_get.assert_called_with(expected_url,**kwargs)        
```

## Mocking a function with a return value

The previous works fine when we just want to ignore a call, but what if we want to use the return value of the API call for further testing?

We can do this using the `Mock` class from `unittest.mock`:

In [ ]:
from unittest.mock import Mock

# dict with some fake issues
fake_issues = [
    {"number": 1,
    "state": "open",
    "title": "Test issue 1"},
    {"number": 2,
    "state": "closed",
    "title": "Test issue 2"},
]

class TestCase:
    # Creating a Mock object lets us add a return value for mocked calls
    @patch("requests.get", new=Mock(return_value=fake_issues))
    def test_return_values(self):
        ### Check that we get the mock values back
        test_repo = "d/test"
        issues = get_repo_issues(test_repo)
        requests.get.assert_called_once()
        assert len(issues) == 2
        
ipytest.run()

But `get_repo_issues` doesn't return a list of issues, it returns an API request response. So we can do better!

## Returning a dummy object

Instead of just returning our dummy issue list, we should be returning a whole dummy response. This will let us use the response exactly like we would a real API call:

In [ ]:
fake_issues = [
    {"number": 1,
    "state": "open",
    "title": "Test issue 1"},
    {"number": 2,
    "state": "closed",
    "title": "Test issue 2"},
]

def dummy_response(repo, **kwargs):
    ### Create a fake response from the website
    dummy_response = requests.Response()
    dummy_response.status_code = 200
    dummy_response.json = Mock(return_value=fake_issues)
    return dummy_response
    

class TestCase:
    # We now use side_effect to bind get to our defined function
    @patch("requests.get", new=Mock(side_effect=dummy_response))
    def test_return_values(self):
        ### Check that we get the mock object back
        test_repo = "d/test"
        response = get_repo_issues(test_repo)
        requests.get.assert_called_once()
        
        # Check that we get the appropriate status code
        assert response.status_code == 200
          
        # Check that we still extract the dummy issues as required
        issues = response.json()
        assert len(issues) == 2 
        
ipytest.run()

## Testing our compound function

We can now test out our compound `print_repo_issues` function! 

We will want to test that the function prints the output we expect, so we will use pytest's built-in `capsys` fixture to read the stdout.

Let's make a new test for that:

In [ ]:
class TestCase:
    @patch("requests.get", new=Mock(side_effect=dummy_response))
    def test_print_repo_issues(self, capsys):
        ### Test that our compound function prints what we expect with our dummy issues
        test_repo = "d/test"
        
        print_repo_issues(test_repo)

        # Read the std out and err
        captured = capsys.readouterr()
        
        # Check that the expected table was printed
        expected_output = ( "# | ? | Name\n"
                            "------------\n"
                            "1 | ✓ | Test issue 1\n"
                            "2 | x | Test issue 2")
        assert expected_output in captured.out
        
ipytest.run()

## Do we really need to patch every test?

Sort of yes, but practically no.
We can use a [`fixture`](## "A pytest decorator that defines a method that can be automatically called ") with the  `autouse` feature.

In [ ]:
@pytest.fixture(autouse=True)
def apply_get_patch(): 
    ### A fixture that mocks out the get function of requests
    requests.get = Mock(side_effect=dummy_response)
    
class TestCase:
    def test_print_repo_with_fixture(self, capsys):
        ### The same test as previously but now with an automatic fixture for mocking get
        test_repo = "d/test"
        
        print_repo_issues(test_repo)

        # Read the std out and err
        captured = capsys.readouterr()
        
        # Check that the expected table was printed
        expected_output = ( "# | ? | Name\n"
                            "------------\n"
                            "1 | ✓ | Test issue 1\n"
                            "2 | x | Test issue 2")
        assert expected_output in captured.out

    def test_return_values(self):
        ### Check that we get the mock object back
        test_repo = "d/test"
        response = get_repo_issues(test_repo)
        requests.get.assert_called_once()
        
        # Check that we get the appropriate status code
        assert response.status_code == 200
          
        # Check that we still extract the dummy issues as required
        issues = response.json()
        assert len(issues) == 2
        
ipytest.run()

## Patching a `patch` request

In exactly the same way, we can mock the `patch` request:

In [ ]:
def dummy_patch_response(url, **kwargs):
    ### Make a dummy response for a patch request
    dummy_response = requests.Response()
    dummy_response.status_code = 200
    # Get the fake issue from our list based on issue number
    issue_number = int(url.split("/issues/")[-1])
    return_issue = fake_issues[issue_number-1]
    # Update its entry based on the data passed to patch
    for kwarg_name, kwarg_value in eval(kwargs["data"]).items():
        if kwarg_name in return_issue:
            return_issue[kwarg_name] = kwarg_value
    dummy_response.json = Mock(return_value=return_issue)
    return dummy_response

class TestCase:
    @patch("requests.patch", new=Mock(side_effect=dummy_patch_response))
    def test_patch_request(self):
        ### Test the patch request
        test_repo = "d/leggat"
        issue_num = 2
        
        payload = {"state":"open"}
        response = patch_issue(repo, issue_num, payload)
        assert response.json()["state"] == "open"
        
        payload = {"state":"closed"}
        response = patch_issue(repo, issue_num, payload)
        assert response.json()["state"] == "closed"
        
ipytest.run()

This works, but we have a lot of repetition in the dummy response code.
We should refactor it so that it only requires one function:

In [ ]:
def response_json_patch(url, data):
    # Get the issue information based on url information
    issue_number = int(url.split("/issues/")[-1])
    return_issue = fake_issues[issue_number-1]
    # Update its entry based on the data passed to patch
    for data_name, data_value in data.items():
        if data_name in return_issue:
            return_issue[data_name] = data_value
    # return a mock with this as a return value
    return Mock(return_value=return_issue)

def dummy_response(url, type_of_request="get", **kwargs):
    ### Create a fake response from the website
    dummy_response = requests.Response()
    dummy_response.status_code = 200
    if type_of_request == "get":
        dummy_response.json = Mock(return_value=fake_issues)
    if type_of_request == "patch":
        dummy_response.json = response_json_patch(url,eval(kwargs["data"]))
    return dummy_response

@pytest.fixture(autouse=True)
def apply_api_patches():
    ### A fixture that mocks out the get function of requests
    print("Still doing this one")
    requests.get   = Mock(side_effect=lambda url, **kwargs: dummy_response(url, type_of_request="get", **kwargs))
    requests.patch = Mock(side_effect=lambda url, **kwargs: dummy_response(url, type_of_request="patch", **kwargs))

class TestCase:
    def test_patch_request(self):
        ### Test the patch request
        test_repo = "d/leggat"
        issue_num = 2
        
        payload = {"state":"open"}
        response = patch_issue(test_repo, issue_num, payload)
        assert response.json()["state"] == "open"
        
        payload = {"state":"closed"}
        response = patch_issue(test_repo, issue_num, payload)
        assert response.json()["state"] == "closed"
        
ipytest.run()

This is all very well and good, but right now each API call made just does its thing and then the state of the system forgets.

What if we want to perform a sequence of tests, and retain the states we've altered?

## Fake API objects


In [ ]:
import copy

class Fake_API:
    """
    A wrapper for a set of API interactions 
    """

    def __init__(self):
        ### Initialisation
        self.fake_issues = []
        
    def set_fake_issues(self, issues):
        self.fake_issues = copy.deepcopy(issues)
        
    def patch_json(self, url, data):
        # Get the issue information based on url information
        issue_number = int(url.split("/issues/")[-1]) - 1 # -1 for zero index
        # Patch the issue in self.fake_issues
        for data_name, data_value in data.items():
            if data_name in self.fake_issues[issue_number]:
                self.fake_issues[issue_number][data_name] = data_value
        # return a mock with this as a return value
        return Mock(return_value=self.fake_issues[issue_number])
        
        
    def dummy_response(self, url, type_of_request="get", **kwargs):
        ### Create a fake response from the website
        dummy_response = requests.Response()
        dummy_response.status_code = 200
        if type_of_request == "get":
            dummy_response.json = Mock(return_value=self.fake_issues)
        if type_of_request == "patch":
            dummy_response.json = self.patch_json(url,eval(kwargs["data"]))
        return dummy_response

This retains the state of our dummy issues. Updating them in a patch request will now linger for future gets (in the same test!).

Let's set up our mock patches:

In [ ]:
@pytest.fixture()
def fake_api():
    fake_api = Fake_API()
    fake_api.set_fake_issues(fake_issues[:])
    requests.get   = Mock(side_effect=lambda url, **kwargs: fake_api.dummy_response(url, type_of_request="get", **kwargs))
    requests.patch = Mock(side_effect=lambda url, **kwargs: fake_api.dummy_response(url, type_of_request="patch", **kwargs))

In [ ]:
# Not sure if I need this panel.

@pytest.fixture(autouse=False)
def apply_api_patches():
    return

@pytest.fixture(autouse=False)
def apply_get_patch():
    return

## Using the fake API object in a test

We can call this fixture as an argument to a test:

In [ ]:
class TestCase:
    def test_post_changes_get(self, fake_api):
        ### Test the state of the issues we get from the API before and after a patch
        test_repo = "d/leggat"
        response = get_repo_issues(test_repo)
        requests.get.assert_called_once()
        
        # Initially the issue is closed
        assert response.json()[1]["state"] == "closed"
        
        # Open the issue
        payload = {"state":"open"}
        patch_respone = patch_issue(test_repo, 2, payload)
        
        # Now when we get the issues again, it is open
        response = get_repo_issues(test_repo)
        assert response.json()[1]["state"] == "open"
        
ipytest.run()

This is a very 'toy' example, but this is a useful strategy for testing when your code wants to make changes to a remote resource.

# Conclusions

Different approaches to testing API calls in python:

1) Use `vcr` cassettes to save real data and use it in the tests.

2) Use mocks to block API calls.

3) Use dummy objects to mimic calls with a known output.

4) Create fake API objects to persist changes that our code makes in the remote resource.

5) Combine all - a fake API object that persists state that initialises from a cassette!

# Things I've used to make this presentation

- `requests` module in `Python` handles API calls.
  - We can also use GraphQL for fancier formatting of API calls, but that was beyond the scope of this presentation.
- RISE (Reveal.js Jupyter/iPython Slideshow Extension) that's right, terrible acronyms aren't just for physicists anymore! (Sadly this plug-in looks kinda dead :'( )
- httpbin.org - for dummy API calls
- ipytest - lets you run pytest within jupyter cells

![image](./images/animals_2.png)